# 01 · Exploración de los datos

Este notebook muestra de forma simple las series que descarga el proyecto desde la API del Banco Central de Chile.

**Antes de ejecutarlo**, descarga los datos desde la terminal (en la raíz del proyecto):

```bash
python extraccion/E_BCCH.py
```

El notebook lee automáticamente **todos** los CSV de `data/raw/`, así que al agregar nuevas series al script de extracción aparecerán aquí sin cambiar el código.

## 1. Librerías y ubicación de los datos

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Busca la raíz del proyecto (la carpeta que contiene requirements.txt),
# así el notebook funciona sin importar desde dónde se abra.
RAIZ = Path.cwd()
while not (RAIZ / "requirements.txt").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

CARPETA_DATOS = RAIZ / "data" / "raw"
archivos = sorted(CARPETA_DATOS.glob("*.csv"))

if not archivos:
    raise FileNotFoundError(
        f"No hay CSV en {CARPETA_DATOS}. Ejecuta primero: python extraccion/E_BCCH.py"
    )

print(f"Se encontraron {len(archivos)} series en {CARPETA_DATOS.relative_to(RAIZ)}:")
for a in archivos:
    print("  -", a.name)

Se encontraron 2 series en data/raw:
  - dolar_observado.csv
  - tpm.csv


## 2. Carga de las series

Cada CSV tiene las columnas `fecha`, `valor`, `serie` y `codigo`. Se juntan todas en una sola tabla en formato **largo** (una fila por serie y fecha), que es el mismo formato que usaremos después en la base de datos SQL.

In [2]:
datos = pd.concat(
    [pd.read_csv(a, parse_dates=["fecha"]) for a in archivos],
    ignore_index=True,
)

datos.head()

,fecha,valor,serie,codigo
0,2000-01-01,520.447619,dolar_observado,F073.TCO.PRE.HIST.M
1,2000-02-01,512.850952,dolar_observado,F073.TCO.PRE.HIST.M
2,2000-03-01,504.380435,dolar_observado,F073.TCO.PRE.HIST.M
3,2000-04-01,508.099474,dolar_observado,F073.TCO.PRE.HIST.M
4,2000-05-01,521.663182,dolar_observado,F073.TCO.PRE.HIST.M


## 3. Ficha resumen de cada serie

Para cada serie: su código en la BDE, el período que cubre, cuántas observaciones tiene, cuántas faltan y su último valor disponible.

In [ ]:
def resumir(grupo):
    grupo = grupo.sort_values("fecha")
    ultimo = grupo.dropna(subset=["valor"]).iloc[-1]
    return pd.Series({
        "codigo": grupo["codigo"].iloc[0],
        "desde": grupo["fecha"].min().strftime("%Y-%m"),
        "hasta": grupo["fecha"].max().strftime("%Y-%m"),
        "observaciones": len(grupo),
        "faltantes": int(grupo["valor"].isna().sum()),
        "ultimo_valor": round(ultimo["valor"], 2),
        "fecha_ultimo_valor": ultimo["fecha"].strftime("%Y-%m"),
    })

resumen = datos.groupby("serie").apply(resumir, include_groups=False)
resumen

## 4. Vista de los datos

La misma información en formato **ancho**: una columna por serie y una fila por mes. Se muestran los últimos 12 meses.

In [ ]:
tabla = datos.pivot(index="fecha", columns="serie", values="valor")
tabla.tail(12).round(2)

## 5. Evolución de cada serie

In [ ]:
series = tabla.columns
fig, ejes = plt.subplots(
    nrows=len(series), ncols=1,
    figsize=(10, 2.6 * len(series)),
    sharex=True, squeeze=False,
)

for eje, nombre in zip(ejes[:, 0], series):
    s = tabla[nombre].dropna()
    eje.plot(s.index, s.values, color="#1f5f99", linewidth=1.4)
    eje.set_title(f"{nombre}  ({resumen.loc[nombre, 'codigo']})", loc="left", fontsize=11)
    eje.grid(alpha=0.3)
    eje.spines[["top", "right"]].set_visible(False)

ejes[-1, 0].set_xlabel("Fecha")
fig.suptitle("Series del proyecto · Banco Central de Chile", fontsize=13, x=0.01, ha="left")
fig.tight_layout()
plt.show()

## Próximos pasos

- Agregar el resto de las series al diccionario `SERIES` de `extraccion/E_BCCH.py` y volver a ejecutar este notebook.
- Cargar estos datos a la base de datos SQLite (tablas de catálogo, observaciones y clasificación temática).